# Lab 02 Extra - Banco de Dados Northwind
**Disciplina:** Extração e Preparação de Dados | **Professor:** Luis Aramis

Este é um notebook extra para praticar SQL com um banco de dados clássico: o **Northwind**.
Ele simula uma importadora/exportadora de alimentos gourmet.

## 1. Setup e Download
Vamos baixar o `northwind.db` e conectar o SQLAlchemy.

In [2]:
import pandas as pd
from sqlalchemy import create_engine
import os
import urllib.request

# Download do northwind.db
if not os.path.exists('northwind.db'):
    # URL do repositório jpwhite3/northwind-SQLite3
    url = 'https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db'
    urllib.request.urlretrieve(url, 'northwind.db')
    print('Banco Northwind baixado com sucesso!')

engine = create_engine('sqlite:///northwind.db')
print('Conexão estabelecida!')

Conexão estabelecida!


## 2. Mapa do Banco
Quais tabelas temos aqui?

In [3]:
query_tabelas = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql(query_tabelas, engine)


,name
0,Categories
1,CustomerCustomerDemo
2,CustomerDemographics
3,Customers
4,EmployeeTerritories
5,Employees
6,Order Details
7,Orders
8,Products
9,Regions


## 3. Consultas Básicas
1. Liste os 5 produtos mais caros (`Products`).
2. Liste todos os clientes (`Customers`) que moram no 'Brazil'.

In [4]:
# Produtos mais caros
query_produtos_caros = """
SELECT ProductName, UnitPrice
FROM Products
ORDER BY UnitPrice DESC
LIMIT 5;
"""

pd.read_sql(query_produtos_caros, engine)


,ProductName,UnitPrice
0,Côte de Blaye,263.50
1,Thüringer Rostbratwurst,123.79
2,Mishi Kobe Niku,97.00
3,Sir Rodney's Marmalade,81.00
4,Carnarvon Tigers,62.50


In [5]:
# Clientes do Brasil
query_clientes_brasil = """
SELECT CustomerID, CompanyName, City, Country
FROM Customers
WHERE Country = 'Brazil'
ORDER BY CompanyName;
"""

pd.read_sql(query_clientes_brasil, engine)


,CustomerID,CompanyName,City,Country
0,COMMI,Comércio Mineiro,Sao Paulo,Brazil
1,FAMIA,Familia Arquibaldo,Sao Paulo,Brazil
2,GOURL,Gourmet Lanchonetes,Campinas,Brazil
3,HANAR,Hanari Carnes,Rio de Janeiro,Brazil
4,QUEDE,Que Delícia,Rio de Janeiro,Brazil
5,QUEEN,Queen Cozinha,Sao Paulo,Brazil
6,RICAR,Ricardo Adocicados,Rio de Janeiro,Brazil
7,TRADH,Tradição Hipermercados,Sao Paulo,Brazil
8,WELLI,Wellington Importadora,Resende,Brazil


## 4. JOIN: Pedidos e Clientes
Vamos ver quem fez quais pedidos.
Tabelas: `Orders` e `Customers`.
Chave de ligação: `CustomerID`.

In [6]:
query_pedidos_clientes = """
SELECT o.OrderID,
       o.OrderDate,
       c.CustomerID,
       c.CompanyName,
       c.Country
FROM Orders o
JOIN Customers c
  ON o.CustomerID = c.CustomerID
ORDER BY o.OrderDate;
"""

pd.read_sql(query_pedidos_clientes, engine)


,OrderID,OrderDate,CustomerID,CompanyName,Country
0,18429,2012-07-10 15:40:46,GREAL,Great Lakes Food Market,USA
1,25506,2012-07-10 20:28:57,RICAR,Ricardo Adocicados,Brazil
2,26048,2012-07-11 01:09:16,LONEP,Lonesome Pine Restaurant,USA
3,16958,2012-07-11 20:26:28,FOLKO,Folk och fä HB,Sweden
4,25877,2012-07-11 21:17:36,NORTS,North/South,UK
...,...,...,...,...,...
16277,11298,2023-10-25 13:00:29,WHITC,White Clover Markets,USA
16278,23676,2023-10-26 06:28:53,ROMEY,Romero y tomillo,Spain
16279,13789,2023-10-27 06:38:44,MORGK,Morgenstern Gesundkost,Germany
16280,25677,2023-10-27 18:17:38,BOLID,Bólido Comidas preparadas,Spain


## 5. JOIN Triplo: Detalhes do Pedido
O que tem dentro do pedido 10248?
Caminho: `OrderDetails` -> `Products`.

In [8]:
query_pedido_10248 = """
SELECT od.OrderID,
       od.ProductID,
       p.ProductName,
       od.Quantity,
       od.UnitPrice,
       ROUND(od.Quantity * od.UnitPrice * (1 - od.Discount), 2) AS TotalItem
FROM "Order Details" od
JOIN Products p
  ON od.ProductID = p.ProductID
WHERE od.OrderID = 10248
ORDER BY TotalItem DESC;
"""

pd.read_sql(query_pedido_10248, engine)


,OrderID,ProductID,ProductName,Quantity,UnitPrice,TotalItem
0,10248,72,Mozzarella di Giovanni,5,34.8,174.0
1,10248,11,Queso Cabrales,12,14.0,168.0
2,10248,42,Singaporean Hokkien Fried Mee,10,9.8,98.0


## 6. Desafio: Total de Vendas por Categoria
Descubra qual Categoria de produtos (`Categories`) gerou mais receita.
Dica: Você vai precisar ligar `Categories` -> `Products` -> `Order Details`.

In [9]:
# Seu código aqui
query_receita_categoria = """
SELECT c.CategoryName,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 2) AS ReceitaTotal
FROM Categories c
JOIN Products p
  ON c.CategoryID = p.CategoryID
JOIN "Order Details" od
  ON p.ProductID = od.ProductID
GROUP BY c.CategoryID, c.CategoryName
ORDER BY ReceitaTotal DESC;
"""

pd.read_sql(query_receita_categoria, engine)


,CategoryName,ReceitaTotal
0,Beverages,92163184.18
1,Confections,66337803.06
2,Meat/Poultry,64881147.97
3,Dairy Products,58018116.78
4,Condiments,55795126.78
5,Seafood,49921604.17
6,Produce,32701119.88
7,Grains/Cereals,28568530.34
